### 目的
ダミーの商品購買データを作る

### 背景
サンドボックス環境なので本番データや機密データは入れてはいけない

### 処理概要
顧客と営業部のデータを生成し、売上データを計算して結合し  


In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

# 顧客名とIDを生成
customer_names = [f"顧客{chr(65 + i)}" for i in range(300)]
customer_ids = [f"{i+1:07d}" for i in range(300)]

# 営業部名とIDを生成
sales_dept_names = [f"営業部{chr(65 + i)}" for i in range(12)]
sales_dept_ids = [f"{i+1:03d}" for i in range(12)]
company_name = "ダミーカンパニーオー"

# 営業部のデータフレームを作成
sales_dept_df = pd.DataFrame({
    "sales_dept_name": sales_dept_names,
    "sales_dept_id": sales_dept_ids,
    "company": [company_name] * 12
})

# 顧客ごとにランダムに営業部IDを割り当て
np.random.seed(42)
assigned_sales_dept_ids = np.random.choice(sales_dept_ids, size=300)

# 顧客基準の売上データを生成
customer_revenue_data = {
    'date': pd.date_range(start='2023-01-01', periods=12, freq='M').tolist() * 300,
    'product': ['kakuu365'] * 12 * 300,
    'customer_names': np.repeat(customer_names, 12),
    'customer_ids': np.repeat(customer_ids, 12),
    'monthly_fee': [1000] * 12 * 300,
    'user_count': np.random.randint(1, 100, 12 * 300),
    'sales_dept_id': np.repeat(assigned_sales_dept_ids, 12)
}

# 売上を計算
customer_revenue_data['revenue'] = np.array(customer_revenue_data['monthly_fee']) * np.array(customer_revenue_data['user_count'])

# Pandas DataFrameを作成
p_customer_revenue_df = pd.DataFrame(customer_revenue_data)

# Spark DataFrameに変換
customer_revenue_df = spark.createDataFrame(p_customer_revenue_df)

# 顧客テーブルを作成
customer_df = pd.DataFrame({
    "customer_name": customer_names,
    "customer_id": customer_ids
})
customer_spark_df = spark.createDataFrame(customer_df)
customer_spark_df.write.mode("overwrite").saveAsTable("saito_catalog.demo_data.customers")

# 営業部テーブルを作成
sales_dept_spark_df = spark.createDataFrame(sales_dept_df)
sales_dept_spark_df.write.mode("overwrite").saveAsTable("saito_catalog.demo_data.sales_depts")

# 売上テーブルを作成
customer_revenue_df.write.mode("overwrite").saveAsTable("saito_catalog.demo_data.revenues")

# 確認用
display(customer_spark_df.limit(10))
display(sales_dept_spark_df.limit(10))
display(customer_revenue_df.limit(10))